# Native-paper visual attack reproduction workbench

28 October 2026. **New protocol/source-audit notebook, not a completed attack run.**
The earlier presentation documents our common-benchmark adaptations; this workbench
tracks each original paper's method, inputs, settings and evaluation separately.
Scope is twelve registered readings: foundations, VLM attacks, benchmark, defenses and survey.
“Covered in the ledger” never means “experiment successfully reproduced.”

Default execution performs CPU metadata/source checks only. No model weights, API calls,
new synthetic data or scientific attack outputs are generated. Original-run result fields
start as `NOT_RUN`. Changing checkpoint, data, objective or scoring creates an adaptation.

## 1. Evidence lanes and completion rules

**Native-paper lane:** original supported checkpoint, data/targets, splits, preprocessing,
objective, optimizer, artifact encoding, decoding and judge. Reduced scale must be explicit.

**Matched-comparison lane:** one held-out benchmark and scorer across mechanisms; our
previous Qwen/Smol results belong here. It supports comparison but does not replicate
every native paper. No newly generated replacement corpus is allowed in either lane.

A faithful result requires an auditable artifact, not a paper title or matching algorithm name.
Full paper table replication remains pending where original releases/compute are unavailable.

### Architecture learning map

Learn: backpropagation/cross-entropy → transformer attention → ViT → CLIP →
causal language decoder → multimodal connectors → full VLM attack path.

| Component | Mathematical object to understand | Why it matters here |
|---|---|---|
| Backpropagation | $\nabla_x L$ and the chain rule | Frozen weights still permit image gradients |
| Attention | $\mathrm{softmax}(QK^T/\sqrt{d_k})V$ | Token interactions in vision and language |
| ViT | Image patches → embeddings + positions → encoder tokens | Pixels become features the connector can use |
| CLIP | Image/text embedding spaces and their distance/similarity | Jailbreak-in-Pieces targets embeddings |
| Causal decoder | $p(y_t\mid y_{<t},q,z)$ | Qi/BAP optimize sequence likelihood |
| Connector | $z=C(E(P(x)))$ | Projection/merging/Q-Former paths differ across VLMs |

For pixel attacks the flow is image $x$ → preprocessing $P$ → encoder $E$ → connector
$C$ → decoder → loss. The gradient travels backward through that path:
$\nabla_x L=J_P^T J_E^T J_C^T\nabla_z L$.
For an encoder-only attack, stop the objective at the embedding; no decoder gradient is needed.

Architecture reading list (titles/abstracts checked; learning assignments, not full reviews):
[Transformer](https://arxiv.org/abs/1706.03762),
[ViT](https://arxiv.org/abs/2010.11929),
[CLIP](https://arxiv.org/abs/2103.00020),
[BLIP-2/Q-Former](https://arxiv.org/abs/2301.12597).
For tomorrow prioritize tracing one image through the VLM, explaining targeted NLL,
FGSM/PGD projection, patch support, and the difference between optimization loss and ASR.

In [1]:
from pathlib import Path
import tempfile,sys,hashlib,json
from IPython.display import display,Markdown
REVIEW_ROOT=Path(tempfile.mkdtemp(prefix="vlm_paper_review_PRIVATE_"))
sys.path.insert(0,str(REVIEW_ROOT))
print("Private source review folder:",REVIEW_ROOT)

Private source review folder: /var/folders/fk/rq_1swks4wx8ftwh5hbfrpz00000gn/T/vlm_paper_review_PRIVATE_l_fq6c6c


## 2. Explicit registry and verification code

The next cell contains ordinary Python metadata/functions. It writes a local module
for imports; it does not serialize another source bundle or execute upstream code.

In [2]:
%%writefile {REVIEW_ROOT}/vlm_paper_protocols.py
"""Pinned paper/release audit; no automatic scientific runs or synthetic fallback."""
import hashlib
import json
from pathlib import Path
import subprocess
import sys
import time
import urllib.request

PAPERS = [
    {"id":"fgsm","authors":"Goodfellow et al.","title":"Explaining and Harnessing Adversarial Examples","arxiv":"1412.6572v3","role":"foundation","repo":None,"revision":None,"variants":["untargeted classification"],"datasets":["MNIST"],"blocker":"Original maxout checkpoint/configuration unresolved.","paper_url":"https://arxiv.org/html/1412.6572v3","inspection":"Methods/setup and selected appendices/code inspected; full experiment audit incomplete","reproduction_status":"NOT_RUN"},
    {"id":"pgd","authors":"Madry et al.","title":"Towards Deep Learning Models Resistant to Adversarial Attacks","arxiv":"1706.06083v4","role":"foundation","repo":"MadryLab/cifar10_challenge","revision":"f15682d9f1e26eb47a2d3b371ef8b6c7abcf6276","variants":["MNIST inner attack","CIFAR-10 inner attack"],"datasets":["MNIST","CIFAR-10"],"blocker":"Challenge release defaults are not automatically paper-table settings; original weights/splits not audited.","paper_url":"https://arxiv.org/html/1706.06083v4","inspection":"Methods/setup and selected appendices/code inspected; full experiment audit incomplete","reproduction_status":"NOT_RUN"},
    {"id":"patch","authors":"Brown et al.","title":"Adversarial Patch","arxiv":"1712.09665v2","role":"foundation","repo":None,"revision":None,"variants":["single-model","ensemble","transfer","camouflage","physical"],"datasets":["ImageNet"],"blocker":"Original transformations, image selection and code provenance incomplete; a VLM version changes the task.","paper_url":"https://arxiv.org/html/1712.09665v2","inspection":"Methods/setup and selected appendices/code inspected; full experiment audit incomplete","reproduction_status":"NOT_RUN"},
    {"id":"qi","authors":"Qi et al.","title":"Visual Adversarial Examples Jailbreak Aligned Large Language Models","arxiv":"2306.13213v2","role":"VLM attack","repo":"Unispac/Visual-Adversarial-Examples-Jailbreak-Large-Language-Models","revision":"5fc9883cfa9b4183cf05c6eceef49e449e45e9c9","variants":["MiniGPT-4","InstructBLIP","LLaVA-LLaMA-2","constrained","unconstrained"],"datasets":["released 66-sentence corpus","40 manual requests","RealToxicityPrompts challenging subset"],"blocker":"Original weights/human evaluation/Perspective scoring unrun; A100 configuration differs from T4.","paper_url":"https://arxiv.org/html/2306.13213v2","inspection":"Methods/setup and selected appendices/code inspected; full experiment audit incomplete","reproduction_status":"NOT_RUN"},
    {"id":"pieces","authors":"Shayegani et al.","title":"Jailbreak in pieces: Compositional Adversarial Attacks on Multi-Modal Language Models","arxiv":"2307.14539v2","role":"VLM attack","repo":"erfanshayegani/Jailbreak-In-Pieces","revision":"b7b2bce812adada23f583f63f63e564f84d277c6","variants":["text trigger","OCR trigger","visual trigger","OCR+visual trigger"],"datasets":["paper trigger/scenario collection","AdvBench supplementary subset"],"blocker":"Release supplies examples and encoder code; complete paper collection/splits and version alignment unresolved.","paper_url":"https://arxiv.org/html/2307.14539v2","inspection":"Methods/setup and selected appendices/code inspected; full experiment audit incomplete","reproduction_status":"NOT_RUN"},
    {"id":"hijacks","authors":"Bailey et al.","title":"Image Hijacks: Adversarial Images can Control Generative Models at Runtime","arxiv":"2309.00236v4","role":"VLM attack","repo":"euanong/image-hijacks","revision":"2a4403cf67e8578afdd6e56096ff31524aaa63e7","variants":["specific string","leak-context format","jailbreak behavior","disinformation prompt matching"],"datasets":["Alpaca","AdvBench","demo Alpaca/LLaVA mixture"],"blocker":"Public demo is not full paper experiment suite; jailbreak/prompt-matching configs and original weights unresolved.","paper_url":"https://arxiv.org/html/2309.00236v4","inspection":"Methods/setup and selected appendices/code inspected; full experiment audit incomplete","reproduction_status":"NOT_RUN"},
    {"id":"bap","authors":"Ying et al.","title":"Jailbreak Vision Language Models via Bi-Modal Adversarial Prompt","arxiv":"2406.04031v2","role":"VLM attack","repo":"NY1024/BAP-Jailbreak-Vision-Language-Models-via-Bi-Modal-Adversarial-Prompt","revision":"5350c0bfeb01d9e485603b85a03500280886d275","variants":["query-agnostic visual PGD","feedback-driven image/text search"],"datasets":["paper SafetyBench scenarios","deduplicated AdvBench","query-agnostic corpus"],"blocker":"Original ChatGPT feedback/judging conflicts with open-only scope; original corpus and merged split need frozen artifacts.","paper_url":"https://arxiv.org/html/2406.04031v2","inspection":"Methods/setup and selected appendices/code inspected; full experiment audit incomplete","reproduction_status":"NOT_RUN"},
    {"id":"physical_injection","authors":"Ling et al.","title":"Physical Prompt Injection Attacks on Large Vision-Language Models","arxiv":"2601.17383v1","role":"VLM attack","repo":"2023cghacker/Physical-Prompt-Injection-Attack","revision":"7291d92ee05c4b9079ef824e0b4bd12e8e14316c","variants":["prompt selection","location search","simulation","physical capture"],"datasets":["Embodied City","Habitat","physical vehicle trials"],"blocker":"Upstream states code not fully runnable; physical data/hardware outside initial approved scope.","paper_url":"https://arxiv.org/html/2601.17383v1","inspection":"Methods/setup and selected appendices/code inspected; full experiment audit incomplete","reproduction_status":"NOT_RUN"},
    {"id":"mmj","authors":"Weng et al.","title":"MMJ-Bench: A Comprehensive Study on Jailbreak Attacks and Defenses for Vision Language Models","arxiv":None,"role":"benchmark","repo":"thunxxx/MLLM-Jailbreak-evaluation-MMJ-Bench","revision":"68922de9209478923dbc0a1b4eccb18a69e2c87e","variants":["attack generation","completions","scoring","utility"],"datasets":["HarmBench","MM-Vet"],"blocker":"Original artifact/model/table coverage and GPT-4 utility scoring not reproduced.","paper_url":"https://ojs.aaai.org/index.php/AAAI/article/download/34983/37138","inspection":"Methods/setup and selected appendices/code inspected; full experiment audit incomplete","reproduction_status":"NOT_RUN"},
    {"id":"adashield","authors":"Wang et al.","title":"AdaShield: Safeguarding Multimodal Large Language Models from Structure-based Attack via Adaptive Shield Prompting","arxiv":"2403.09513v1","role":"defense","repo":"SaFo-Lab/AdaShield","revision":"b6d02c6747bd6984a99ee2e1db1e52736a6a05cc","variants":["AdaShield-S","AdaShield-A"],"datasets":["FigStep","MM-SafetyBench QR","MM-Vet"],"blocker":"Generic prefix is not AdaShield; original pool/splits and GPT-4 steps need audit and open-only deviation.","paper_url":"https://arxiv.org/html/2403.09513v1","inspection":"Methods/setup and selected appendices/code inspected; full experiment audit incomplete","reproduction_status":"NOT_RUN"},
    {"id":"jailguard","authors":"Zhang et al.","title":"JailGuard: A Universal Detection Framework for Prompt-based Attacks on LLM Systems","arxiv":"2312.10766v4","role":"defense","repo":"shiningrain/JailGuard","revision":"419e1d3d08f4d02998a11a687f925518d2d5dd0c","variants":["visual mutators","combination policy","divergence detector"],"datasets":["official attack/benign release"],"blocker":"Original policy, encoder, split, threshold and targets not reproduced; no detector run.","paper_url":"https://arxiv.org/html/2312.10766v4","inspection":"Methods/setup and selected appendices/code inspected; full experiment audit incomplete","reproduction_status":"NOT_RUN"},
    {"id":"survey","authors":"Zhang et al.","title":"Adversarial Attacks of Vision Tasks in the Past 10 Years: A Survey","arxiv":"2410.23687v2","role":"survey","repo":None,"revision":None,"variants":[],"datasets":[],"blocker":"Taxonomy/discovery source; not an attack experiment.","paper_url":"https://arxiv.org/html/2410.23687v2","inspection":"Methods/setup and selected appendices/code inspected; full experiment audit incomplete","reproduction_status":"NOT_RUN"},
]

REVIEWED_FILES = [
    {"repo":"Unispac/Visual-Adversarial-Examples-Jailbreak-Large-Language-Models","revision":"5fc9883cfa9b4183cf05c6eceef49e449e45e9c9","path":"README.md","git_blob":"f2468d6882fff91a3a6a6176d1ee10963aa3e252"},
    {"repo":"erfanshayegani/Jailbreak-In-Pieces","revision":"b7b2bce812adada23f583f63f63e564f84d277c6","path":"README.md","git_blob":"6fad07fcc15f576f189e20b4a8b6c814ceeaad64"},
    {"repo":"euanong/image-hijacks","revision":"2a4403cf67e8578afdd6e56096ff31524aaa63e7","path":"README.md","git_blob":"eb3274ab29cd4cae51ac4f2dd79aac33ce17662f"},
    {"repo":"NY1024/BAP-Jailbreak-Vision-Language-Models-via-Bi-Modal-Adversarial-Prompt","revision":"5350c0bfeb01d9e485603b85a03500280886d275","path":"README.md","git_blob":"3fc4b0182e26945ac4f6dee3d9217eb18a07e0fb"},
    {"repo":"2023cghacker/Physical-Prompt-Injection-Attack","revision":"7291d92ee05c4b9079ef824e0b4bd12e8e14316c","path":"README.md","git_blob":"39db3833a94bd88ba775d58f9f1f7ccc9256a22d"},
    {"repo":"thunxxx/MLLM-Jailbreak-evaluation-MMJ-Bench","revision":"68922de9209478923dbc0a1b4eccb18a69e2c87e","path":"README.md","git_blob":"331b103bf7b21ccc7bdc301f74c830b23e2d6a93"},
    {"repo":"SaFo-Lab/AdaShield","revision":"b6d02c6747bd6984a99ee2e1db1e52736a6a05cc","path":"README.md","git_blob":"801e74f78b0a86be05d8f50d45b745953d0d42d8"},
    {"repo":"shiningrain/JailGuard","revision":"419e1d3d08f4d02998a11a687f925518d2d5dd0c","path":"README.md","git_blob":"3f7e0c2d533dca9a25d0066448c5882ebb4a3a54"},
    {"repo":"Unispac/Visual-Adversarial-Examples-Jailbreak-Large-Language-Models","revision":"5fc9883cfa9b4183cf05c6eceef49e449e45e9c9","path":"minigpt_visual_attack.py","git_blob":"55d27fdea7b5e899952ca6646ca49ab4c03eb60b"},
    {"repo":"Unispac/Visual-Adversarial-Examples-Jailbreak-Large-Language-Models","revision":"5fc9883cfa9b4183cf05c6eceef49e449e45e9c9","path":"minigpt_utils/visual_attacker.py","git_blob":"74832621b9f37e128a68d763e32cf91c39e18e76"},
    {"repo":"Unispac/Visual-Adversarial-Examples-Jailbreak-Large-Language-Models","revision":"5fc9883cfa9b4183cf05c6eceef49e449e45e9c9","path":"eval_configs/minigpt4_eval.yaml","git_blob":"54128ba4a62bdf873b9172d846ef4d08b356de72"},
    {"repo":"erfanshayegani/Jailbreak-In-Pieces","revision":"b7b2bce812adada23f583f63f63e564f84d277c6","path":"src/adv_image.py","git_blob":"b9cd5f79bfd087f92481ead5443e10b0e3dd14fa"},
    {"repo":"euanong/image-hijacks","revision":"2a4403cf67e8578afdd6e56096ff31524aaa63e7","path":"experiments/exp_demo_imgs/config.py","git_blob":"d87a7f8f04ff8597a963f78d052b16641fcea9d0"},
    {"repo":"euanong/image-hijacks","revision":"2a4403cf67e8578afdd6e56096ff31524aaa63e7","path":"image_hijacks/data.py","git_blob":"05359db3f60f3bcf15fbf3d671d83e569c44068a"},
    {"repo":"euanong/image-hijacks","revision":"2a4403cf67e8578afdd6e56096ff31524aaa63e7","path":"image_hijacks/config.py","git_blob":"413b62b2772340c8f99094ac0dc4d71d17146497"},
    {"repo":"euanong/image-hijacks","revision":"2a4403cf67e8578afdd6e56096ff31524aaa63e7","path":"image_hijacks/components/loss.py","git_blob":"6834ac42eb58b5586211a7e7701b9083df7add0d"},
    {"repo":"euanong/image-hijacks","revision":"2a4403cf67e8578afdd6e56096ff31524aaa63e7","path":"image_hijacks/components/optim.py","git_blob":"e02e6c89728c75144582d58e627056d402d0ed46"},
    {"repo":"NY1024/BAP-Jailbreak-Vision-Language-Models-via-Bi-Modal-Adversarial-Prompt","revision":"5350c0bfeb01d9e485603b85a03500280886d275","path":"VAP.py","git_blob":"410381b2da3998b024db9ec28ae1d539768e29c9"},
    {"repo":"NY1024/BAP-Jailbreak-Vision-Language-Models-via-Bi-Modal-Adversarial-Prompt","revision":"5350c0bfeb01d9e485603b85a03500280886d275","path":"CoT.py","git_blob":"32c219e78fe5ed16869a8527ca72e96cb34d2aea"},
    {"repo":"NY1024/BAP-Jailbreak-Vision-Language-Models-via-Bi-Modal-Adversarial-Prompt","revision":"5350c0bfeb01d9e485603b85a03500280886d275","path":"Eval.py","git_blob":"36a7f9353dd533e1eba2a15b7958118dae19fa69"},
    {"repo":"MadryLab/cifar10_challenge","revision":"f15682d9f1e26eb47a2d3b371ef8b6c7abcf6276","path":"config.json","git_blob":"613efd2e235b892df14bf084defddfac67f3fe82"},
    {"repo":"MadryLab/mnist_challenge","revision":"3ee3643c4a8c59458d8c191b84027f4a6cbd9580","path":"config.json","git_blob":"af5efb1f138af0217d8891740b62e2b3faa59df3"},
]



def source_url(entry):
    return 'https://raw.githubusercontent.com/'+entry['repo']+'/'+entry['revision']+'/'+entry['path']


def audit_source(entry, private_root):
    """Download one reviewed source file only; verify Git blob identity; never execute it."""
    if entry not in REVIEWED_FILES:
        raise ValueError('Source not in reviewed allowlist')
    root=Path(private_root)
    if 'PRIVATE' not in str(root):
        raise ValueError('Private review folder required')
    dest=root/entry['repo']/entry['path']
    dest.parent.mkdir(parents=True,exist_ok=True)
    if not dest.exists():
        request=urllib.request.Request(source_url(entry),headers={'User-Agent':'VLM-paper-audit'})
        with urllib.request.urlopen(request,timeout=45) as response:
            payload=response.read(2_000_001)
        if len(payload)>2_000_000:raise ValueError('Unexpected large source file')
        dest.write_bytes(payload)
    data=dest.read_bytes()
    blob=hashlib.sha1(b'blob '+str(len(data)).encode()+b'\0'+data).hexdigest()
    if blob!=entry['git_blob']:raise ValueError('Pinned source identity mismatch')
    syntax=dict(status='not_python')
    if entry['path'].endswith('.py'):
        try:
            compile(data,str(dest),'exec');syntax=dict(status='passed')
        except SyntaxError as error:
            syntax=dict(status='invalid_upstream_source',error_type=type(error).__name__,line=error.lineno)
    return dict(repo=entry['repo'],path=entry['path'],revision=entry['revision'],
                git_blob_verified=True,sha256=hashlib.sha256(data).hexdigest(),bytes=len(data),syntax=syntax)


def classify_evidence(record):
    """A name match never promotes a run into faithful reproduction."""
    required=('paper_version','code_revision','checkpoint_hash','processor_hash',
              'dataset_hashes','split_ids_hash','objective_id','optimizer_settings',
              'decoding_settings','judge_protocol','source_result_location',
              'raw_output_manifest_hash','completion_audit')
    missing=[key for key in required if not record.get(key)]
    differences=record.get('deviations',[])
    if missing:return dict(label='UNRESOLVED',missing=missing,deviations=differences)
    if differences:return dict(label='ADAPTATION',missing=[],deviations=differences)
    if record.get('scale_reduced'):return dict(label='REDUCED_SCALE_CANDIDATE',requires='review of retained methodology')
    return dict(label='FAITHFUL_CANDIDATE',requires='independent protocol/result review; not automatic validation')


def reference_command(paper_id, private_checkout, output_dir):
    """Only entrypoints actually inspected. Separate environments are still required."""
    paper=next(p for p in PAPERS if p['id']==paper_id)
    checkout=Path(private_checkout)
    if 'PRIVATE' not in str(checkout) or 'PRIVATE' not in str(output_dir):
        raise ValueError('Private source/output folders required')
    revision=subprocess.check_output(['git','rev-parse','HEAD'],cwd=checkout,text=True).strip()
    if revision!=paper['revision']:raise ValueError('Original source revision differs')
    if paper_id=='qi':
        # Upstream parser uses underscores. It hardcodes num_iter=5000 and batch_size=8.
        return [sys.executable,'minigpt_visual_attack.py','--cfg_path',
                'eval_configs/minigpt4_eval.yaml','--gpu_id','0','--n_iters','5000',
                '--constrained','--eps','16','--alpha','1','--save_dir',str(output_dir)],checkout
    if paper_id=='pieces':
        # Upstream demo: CLIP 336, released trigger/white image, Adam .1, 5000 updates.
        return [sys.executable,'adv_image.py'],checkout/'src'
    raise ValueError('No fully reviewed runnable original entrypoint for this paper')


def run_reference(paper_id, private_checkout, output_dir, approval):
    """Never called by default. Missing scientific/compute approvals block model execution."""
    required=('owner_approved','amendment_approved','source_execution_approved',
              'original_weights_verified','original_data_verified','environment_verified',
              'evaluation_protocol_frozen','compute_profile_verified')
    missing=[key for key in required if approval.get(key) is not True]
    if missing:raise ValueError('Unmet original-run gates: '+', '.join(missing))
    command,cwd=reference_command(paper_id,private_checkout,output_dir)
    budget=approval.get('wall_seconds')
    if not isinstance(budget,int) or budget<=0:raise ValueError('Explicit wall-time budget required')
    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)
    clock=out/'reference_clock_PRIVATE.json'
    if clock.exists():
        raise ValueError('Existing attempt preserved; do not reset or replay its clock')
    state=dict(paper=paper_id,started=time.time(),wall_seconds=budget,status='running',
               code_revision=next(p['revision'] for p in PAPERS if p['id']==paper_id))
    clock.write_text(json.dumps(state))
    try:
        with (out/'execution_PRIVATE.log').open('w') as log:
            completed=subprocess.run(command,cwd=cwd,stdout=log,stderr=subprocess.STDOUT,timeout=budget)
        state.update(exit_code=completed.returncode,status='process_finished_requires_result_audit')
    except subprocess.TimeoutExpired:state['status']='budget_stopped_incomplete'
    finally:
        state['finished']=time.time();clock.write_text(json.dumps(state))
    return state


Writing /var/folders/fk/rq_1swks4wx8ftwh5hbfrpz00000gn/T/vlm_paper_review_PRIVATE_l_fq6c6c/vlm_paper_protocols.py


In [3]:
from vlm_paper_protocols import PAPERS,REVIEWED_FILES,audit_source,classify_evidence,reference_command,run_reference
table=["| Study | Role | Native data | Reproduction status |","|---|---|---|---|"]
for p in PAPERS:
    table.append("| "+p["id"]+" | "+p["role"]+" | "+", ".join(p["datasets"])+" | "+p["reproduction_status"]+" |")
display(Markdown("\n".join(table)))
print("Reviewed source/config files:",len(REVIEWED_FILES))

| Study | Role | Native data | Reproduction status |
|---|---|---|---|
| fgsm | foundation | MNIST | NOT_RUN |
| pgd | foundation | MNIST, CIFAR-10 | NOT_RUN |
| patch | foundation | ImageNet | NOT_RUN |
| qi | VLM attack | released 66-sentence corpus, 40 manual requests, RealToxicityPrompts challenging subset | NOT_RUN |
| pieces | VLM attack | paper trigger/scenario collection, AdvBench supplementary subset | NOT_RUN |
| hijacks | VLM attack | Alpaca, AdvBench, demo Alpaca/LLaVA mixture | NOT_RUN |
| bap | VLM attack | paper SafetyBench scenarios, deduplicated AdvBench, query-agnostic corpus | NOT_RUN |
| physical_injection | VLM attack | Embodied City, Habitat, physical vehicle trials | NOT_RUN |
| mmj | benchmark | HarmBench, MM-Vet | NOT_RUN |
| adashield | defense | FigStep, MM-SafetyBench QR, MM-Vet | NOT_RUN |
| jailguard | defense | official attack/benign release | NOT_RUN |
| survey | survey |  | NOT_RUN |

Reviewed source/config files: 22


**Interpretation:** all native-paper runs remain NOT_RUN. The registry describes
what must be reproduced, not invented results. The prior 136 cases do not fill these rows.

## Explaining and Harnessing Adversarial Examples

Goodfellow et al.. [1412.6572v3](https://arxiv.org/html/1412.6572v3).

**Inspection:** Methods/setup and selected appendices/code inspected; full experiment audit incomplete.

**Objective:** maximize a classification loss under an infinity-norm bound:
$\delta^*=\epsilon\,\mathrm{sign}(\nabla_x\ell(f(x),y))$ from a first-order approximation.
Here $x$ is the original pixel tensor, $y$ its class label and $\epsilon$ the declared scale.
The original classification setup is not a generative VLM safety experiment.
Recover its actual trained checkpoint and preprocessing before reproducing a table.
Targeted token-NLL minimization changes the task and reverses the update sign.

**Versions to account for:** untargeted classification.

**Native inputs:** MNIST.

**Unresolved:** Original maxout checkpoint/configuration unresolved.

**Our original-paper result:** NOT_RUN.

## Towards Deep Learning Models Resistant to Adversarial Attacks

Madry et al.. [1706.06083v4](https://arxiv.org/html/1706.06083v4).

**Inspection:** Methods/setup and selected appendices/code inspected; full experiment audit incomplete.

**Objective:** the inner maximization
$\max_{\delta\in S}\ell(f_\theta(x+\delta),y)$, with fixed weights for attack evaluation.
The update is $x_{t+1}=\Pi_{[0,1]\cap B_\infty(x,\epsilon)}
(x_t+\alpha\,\mathrm{sign}(\nabla\ell))$ when pixels use that scale.
The paper's outer weight-minimization is excluded by our no-training scope.
Select a particular original checkpoint/configuration; do not merge the MNIST and
CIFAR challenge defaults or assume a release config is every paper-table attack.

**Versions to account for:** MNIST inner attack, CIFAR-10 inner attack.

**Native inputs:** MNIST, CIFAR-10.

[Official source snapshot](https://github.com/MadryLab/cifar10_challenge/tree/f15682d9f1e26eb47a2d3b371ef8b6c7abcf6276); revision `f15682d9f1e26eb47a2d3b371ef8b6c7abcf6276`.

**Unresolved:** Challenge release defaults are not automatically paper-table settings; original weights/splits not audited.

**Our original-paper result:** NOT_RUN.

## Adversarial Patch

Brown et al.. [1712.09665v2](https://arxiv.org/html/1712.09665v2).

**Inspection:** Methods/setup and selected appendices/code inspected; full experiment audit incomplete.

**Objective:** $\max_p\mathbb E_{x,t,l}\log P(\hat y\mid A(p,x,l,t))$.
$p$ is the shared replacement patch, $x$ a background, $t$ its transformation and $l$
the placement. The classifier target $\hat y$ is fixed. Training and evaluation
backgrounds/distributions must match the chosen original experiment. A bounded,
per-image square at one location is not this complete universal/physical procedure.

**Versions to account for:** single-model, ensemble, transfer, camouflage, physical.

**Native inputs:** ImageNet.

**Unresolved:** Original transformations, image selection and code provenance incomplete; a VLM version changes the task.

**Our original-paper result:** NOT_RUN.

## Visual Adversarial Examples Jailbreak Aligned Large Language Models

Qi et al.. [2306.13213v2](https://arxiv.org/html/2306.13213v2).

**Inspection:** Methods/setup and selected appendices/code inspected; full experiment audit incomplete.

**Objective:** minimize corpus sequence NLL
$J(x')=-\sum_j\log p_\theta(a_j\mid x',\varnothing)$ under the selected image bound.
$a_j$ are **released** corpus entries, not newly generated continuations.
Native route: original MiniGPT-4/Vicuna checkpoint → original processor → corpus sampling
→ constrained/unconstrained optimization → native manual/toxicity evaluation.
Do not replace native evaluation with eight HarmBench questions and call the result faithful.
The inspected entrypoint and inclusive update loop, rather than a flag's apparent name,
determine actual iteration count. Exact source references are in the audit below.

**Versions to account for:** MiniGPT-4, InstructBLIP, LLaVA-LLaMA-2, constrained, unconstrained.

**Native inputs:** released 66-sentence corpus, 40 manual requests, RealToxicityPrompts challenging subset.

[Official source snapshot](https://github.com/Unispac/Visual-Adversarial-Examples-Jailbreak-Large-Language-Models/tree/5fc9883cfa9b4183cf05c6eceef49e449e45e9c9); revision `5fc9883cfa9b4183cf05c6eceef49e449e45e9c9`.

**Unresolved:** Original weights/human evaluation/Perspective scoring unrun; A100 configuration differs from T4.

**Our original-paper result:** NOT_RUN.

## Jailbreak in pieces: Compositional Adversarial Attacks on Multi-Modal Language Models

Shayegani et al.. [2307.14539v2](https://arxiv.org/html/2307.14539v2).

**Inspection:** Methods/setup and selected appendices/code inspected; full experiment audit incomplete.

**Objective:** $\min_{x'}\|E_I(x')-h_{target}\|_2$ with the original CLIP encoder.
$h_{target}$ comes from the selected original text/OCR/visual/combined trigger branch.
This is encoder access, not end-to-end decoder NLL. Use the actual published trigger;
do not substitute an arbitrary feature image. The released script uses an Adam loop,
box clamping and JPEG output; distinguish this released example from the paper's
convergence algorithm and complete four-trigger evaluation. No success is inferred
from embedding proximity alone; downstream behavior has its own evaluation.

**Versions to account for:** text trigger, OCR trigger, visual trigger, OCR+visual trigger.

**Native inputs:** paper trigger/scenario collection, AdvBench supplementary subset.

[Official source snapshot](https://github.com/erfanshayegani/Jailbreak-In-Pieces/tree/b7b2bce812adada23f583f63f63e564f84d277c6); revision `b7b2bce812adada23f583f63f63e564f84d277c6`.

**Unresolved:** Release supplies examples and encoder code; complete paper collection/splits and version alignment unresolved.

**Our original-paper result:** NOT_RUN.

## Image Hijacks: Adversarial Images can Control Generative Models at Runtime

Bailey et al.. [2309.00236v4](https://arxiv.org/html/2309.00236v4).

**Inspection:** Methods/setup and selected appendices/code inspected; full experiment audit incomplete.

**Behavior matching:** minimize
$\sum_{c\in C} CE(B(c),M_\theta(x',c))$ using the native teacher-forced construction.
For prompt matching, $B_p(c)=M_\theta(I,p\Vert c)$ supplies soft output distributions.
$C$ is the context distribution, $p$ the original behavior instruction and $I$ the teacher
image. A single fixed continuation is not full-context prompt matching. Recover
the task-specific splits/validation selection and original checkpoint. Leak-context
experiments evaluate generated text only: no tool/email execution is authorized.

**Versions to account for:** specific string, leak-context format, jailbreak behavior, disinformation prompt matching.

**Native inputs:** Alpaca, AdvBench, demo Alpaca/LLaVA mixture.

[Official source snapshot](https://github.com/euanong/image-hijacks/tree/2a4403cf67e8578afdd6e56096ff31524aaa63e7); revision `2a4403cf67e8578afdd6e56096ff31524aaa63e7`.

**Unresolved:** Public demo is not full paper experiment suite; jailbreak/prompt-matching configs and original weights unresolved.

**Our original-paper result:** NOT_RUN.

## Jailbreak Vision Language Models via Bi-Modal Adversarial Prompt

Ying et al.. [2406.04031v2](https://arxiv.org/html/2406.04031v2).

**Inspection:** Methods/setup and selected appendices/code inspected; full experiment audit incomplete.

**Two stages:** maximize corpus likelihood for a query-agnostic image within
the paper's pixel bound, then iteratively update textual input using previous
model responses and the paper's feedback judge. A fixed candidate list omits that
feedback mechanism. Preserve the original corpus, original merged/deduplicated evaluation
set, iteration cutoff and native decoding. Replacing original ChatGPT with an open
model is a declared methodological adaptation, not a faithful closed-model reproduction.

**Versions to account for:** query-agnostic visual PGD, feedback-driven image/text search.

**Native inputs:** paper SafetyBench scenarios, deduplicated AdvBench, query-agnostic corpus.

[Official source snapshot](https://github.com/NY1024/BAP-Jailbreak-Vision-Language-Models-via-Bi-Modal-Adversarial-Prompt/tree/5350c0bfeb01d9e485603b85a03500280886d275); revision `5350c0bfeb01d9e485603b85a03500280886d275`.

**Unresolved:** Original ChatGPT feedback/judging conflicts with open-only scope; original corpus and merged split need frozen artifacts.

**Our original-paper result:** NOT_RUN.

## Physical Prompt Injection Attacks on Large Vision-Language Models

Ling et al.. [2601.17383v1](https://arxiv.org/html/2601.17383v1).

**Inspection:** Methods/setup and selected appendices/code inspected; full experiment audit incomplete.

**Objective:** select visual instruction content and scene placement that induce
an unauthorized behavior on a legitimate task under the paper's access assumptions.
This is black-box prompt/location selection, not an FGSM objective. Preserve scene,
viewpoint, camera and task protocols and distinguish simulation from photographed
evidence. A released CyberSecEval3 image-marker test does not reproduce this paper.
Upstream's incomplete-run warning remains an explicit blocker.

**Versions to account for:** prompt selection, location search, simulation, physical capture.

**Native inputs:** Embodied City, Habitat, physical vehicle trials.

[Official source snapshot](https://github.com/2023cghacker/Physical-Prompt-Injection-Attack/tree/7291d92ee05c4b9079ef824e0b4bd12e8e14316c); revision `7291d92ee05c4b9079ef824e0b4bd12e8e14316c`.

**Unresolved:** Upstream states code not fully runnable; physical data/hardware outside initial approved scope.

**Our original-paper result:** NOT_RUN.

## MMJ-Bench: A Comprehensive Study on Jailbreak Attacks and Defenses for Vision Language Models

Weng et al.. [AAAI proceedings](https://ojs.aaai.org/index.php/AAAI/article/download/34983/37138).

**Inspection:** Methods/setup and selected appendices/code inspected; full experiment audit incomplete.

**Role:** common-benchmark evaluation pipeline. Generate attack cases, collect target
completions, then label them under the chosen original judge; measure benign utility
separately. This is useful for our later matched comparison, but it intentionally
changes some original attack evaluations. Keep native-paper and common-benchmark
tables separate. Do not count the benchmark itself as another attack.

**Versions to account for:** attack generation, completions, scoring, utility.

**Native inputs:** HarmBench, MM-Vet.

[Official source snapshot](https://github.com/thunxxx/MLLM-Jailbreak-evaluation-MMJ-Bench/tree/68922de9209478923dbc0a1b4eccb18a69e2c87e); revision `68922de9209478923dbc0a1b4eccb18a69e2c87e`.

**Unresolved:** Original artifact/model/table coverage and GPT-4 utility scoring not reproduced.

**Our original-paper result:** NOT_RUN.

## AdaShield: Safeguarding Multimodal Large Language Models from Structure-based Attack via Adaptive Shield Prompting

Wang et al.. [2403.09513v1](https://arxiv.org/html/2403.09513v1).

**Inspection:** Methods/setup and selected appendices/code inspected; full experiment audit incomplete.

**Role:** inference-time defense. Static prompting and adaptive refinement/retrieval
are separate versions. Original attack datasets and benign utility matter. An unrelated
fixed prefix does not reproduce either version automatically; exact prompts/pools must
come from the audited release and remain private. Original API-dependent stages require
an explicit scope decision; no silent open-model replacement.

**Versions to account for:** AdaShield-S, AdaShield-A.

**Native inputs:** FigStep, MM-SafetyBench QR, MM-Vet.

[Official source snapshot](https://github.com/SaFo-Lab/AdaShield/tree/b6d02c6747bd6984a99ee2e1db1e52736a6a05cc); revision `b6d02c6747bd6984a99ee2e1db1e52736a6a05cc`.

**Unresolved:** Generic prefix is not AdaShield; original pool/splits and GPT-4 steps need audit and open-only deviation.

**Our original-paper result:** NOT_RUN.

## JailGuard: A Universal Detection Framework for Prompt-based Attacks on LLM Systems

Zhang et al.. [2312.10766v4](https://arxiv.org/html/2312.10766v4).

**Inspection:** Methods/setup and selected appendices/code inspected; full experiment audit incomplete.

**Role:** detector, not an image optimizer. Produce the paper's input mutations,
query the target, embed/compare responses and apply its divergence decision policy.
Report detection precision/recall/false-positive rates and target behavior separately.
Restore the original policy, sample budget, encoder, thresholds and split before a
reproduction. Flip-aware PGD alone is not JailGuard.

**Versions to account for:** visual mutators, combination policy, divergence detector.

**Native inputs:** official attack/benign release.

[Official source snapshot](https://github.com/shiningrain/JailGuard/tree/419e1d3d08f4d02998a11a687f925518d2d5dd0c); revision `419e1d3d08f4d02998a11a687f925518d2d5dd0c`.

**Unresolved:** Original policy, encoder, split, threshold and targets not reproduced; no detector run.

**Our original-paper result:** NOT_RUN.

## Adversarial Attacks of Vision Tasks in the Past 10 Years: A Survey

Zhang et al.. [2410.23687v2](https://arxiv.org/html/2410.23687v2).

**Inspection:** Methods/setup and selected appendices/code inspected; full experiment audit incomplete.

**Role:** discovery and taxonomy only. Follow citations to original experiments;
the survey cannot verify another paper's exact code/data or supply an independent ASR.

**Versions to account for:** not an attack.

**Native inputs:** citation chains.

**Unresolved:** Taxonomy/discovery source; not an attack experiment.

**Our original-paper result:** NOT_RUN.

## 3. Inspect pinned original source files without model execution

Optional small-file downloads only. Each file is pinned to a reviewed repository commit
and checked against its actual Git blob hash. Source/config files may contain published
prompts, so save them privately; do not print their full contents or commit them.
Compilation checks syntax only, not dependency compatibility or scientific correctness.

In [4]:
FETCH_REVIEWED_SOURCE=False
source_audit=[]
if FETCH_REVIEWED_SOURCE:
    for entry in REVIEWED_FILES:
        source_audit.append(audit_source(entry,REVIEW_ROOT/"upstream_PRIVATE"))
    display(source_audit)
else:
    print("No network/model execution. Set FETCH_REVIEWED_SOURCE=True for the pinned small-file audit only.")

No network/model execution. Set FETCH_REVIEWED_SOURCE=True for the pinned small-file audit only.


## 4. Exact original-code entrypoints — preparation, not execution

The inspected Qi parser requires underscore flags; iteration and batch settings are also
hardcoded inside its source. Its original weights/environment must be present first.
Jailbreak-in-Pieces has a released encoder example with fixed paths/settings.
Other methods remain blocked until their native experiment configurations are recovered.
Separate original environments are required; do not install every legacy dependency
into the current Colab kernel and assume compatibility.

In [5]:
RUN_ORIGINAL=False
PAPER_ID="pieces"
PRIVATE_CHECKOUT=None
PRIVATE_OUTPUT=None
APPROVAL={}  # Populate only from a reviewed, approved scientific/compute protocol.
if RUN_ORIGINAL:
    if PRIVATE_CHECKOUT is None or PRIVATE_OUTPUT is None:
        raise ValueError("Provide verified private original checkout/output paths first")
    outcome=run_reference(PAPER_ID,PRIVATE_CHECKOUT,PRIVATE_OUTPUT,APPROVAL)
    display(outcome)
else:
    print("No original attack run. Scientific and compute gates remain closed.")

No original attack run. Scientific and compute gates remain closed.


**Interpretation:** having a command is not a working reproduction. The original
checkpoint/data/environment must pass review; a process exit does not establish ASR.
An expired or interrupted budget must remain recorded, not silently reset.

## 5. Required record for every actual attempt

Paper/version and target table; source revision; checkpoint and processor hashes;
dataset/target hashes and source IDs; exact split; transforms; objective; optimization
steps/restarts/selection; artifact encoding; decoding; judge; human rubric; attempts,
responses, unresolved labels, costs and failures. Store raw evidence privately.

The fidelity function below reports missing facts instead of manufacturing them.

In [6]:
CURRENT_ORIGINAL_RUN={}  # No scientific run has occurred in this notebook.
display(classify_evidence(CURRENT_ORIGINAL_RUN))

{'label': 'UNRESOLVED',
 'missing': ['paper_version',
  'code_revision',
  'checkpoint_hash',
  'processor_hash',
  'dataset_hashes',
  'split_ids_hash',
  'objective_id',
  'optimizer_settings',
  'decoding_settings',
  'judge_protocol',
  'source_result_location',
  'raw_output_manifest_hash',
  'completion_audit'],
 'deviations': []}

## 6. Evaluation mapping

Classifiers: clean/attacked class accuracy or targeted class success. Generative safety:
the original behavior rubric and sampling protocol. Injection: unauthorized task deviation.
Detection: attack/benign detection metrics and false positives. Benign utility: original
task scorer. These endpoints are not interchangeable and must not be pooled as one ASR.

For a later matched comparison, prespecify all attempted-case denominators, paired clean
controls, unresolved-output handling, request-clustered uncertainty, human review and
benign utility. Keep native-paper scoring as a separate table with its own provenance.

## 7. What can be presented now

The prior presentation contains measured adaptation results. This notebook adds the
original-method ledger and executable source/fidelity checks. No new native-paper ASR
exists. Missing full releases, closed-model dependencies and original hardware needs
are findings about reproducibility, not permission to invent substitutes.

First proposed next stage: released Jailbreak-in-Pieces encoder feasibility on the T4,
after source/model/data revision and memory checks, with a new approved amendment/budget.
This feasibility exercise is not a complete four-trigger, downstream safety reproduction.
Then resolve native Qi checkpoints and missing full configs before expanding runs.

## Appendix: audit findings and exact source locations

# Original-paper reproduction audit

28 October 2026. Scope: the twelve registered core readings, not every visual attack
ever published. This audit advances methods/setup, selected appendix and source inspection;
it is not a declaration that every full text, appendix, release and result table is verified.
No new scientific GPU run was launched for this audit. Earlier 136/144 cases remain
published-source **adaptations**, not retroactively faithful paper reproductions.

Actual source audit: 22 pinned official source/configuration/README files, 120,547 bytes,
all matching their Git blob identities. Python syntax checks identified the two BAP
defects recorded below. This validates source identity and detects syntax failures;
it does not verify model loading, dataset completeness or successful attacks.

## The requested change

Each study needs its native data, model, objective, attack budget, preprocessing,
selection rule, decoding and endpoint. A second common-benchmark comparison can change
these deliberately, but must keep a separate adaptation label. Dataset names alone do
not establish fidelity. A literature review, source audit, executable mechanism and
completed reproduction are four different milestones.

The new notebook is `notebooks/vlm_paper_reproduction.ipynb`; its generator is
`tools/nb_gen/make_paper_reproduction_nb.py`. The executable registry, source hash checks,
fidelity checks and explicitly gated original-code entrypoints live in
`tools/vlm_paper_protocols.py`. No missing original targets or datasets are fabricated.

## Method and data ledger

| Paper and inspected locations | Native procedure / data | Required fidelity work before a result |
|---|---|---|
| Goodfellow et al., [FGSM, §§3–4](https://arxiv.org/html/1412.6572v3) | Untargeted classification loss sign step; MNIST/maxout experiment | Recover exact trained weights and scaling; generative target minimization changes the task |
| Madry et al., [PGD, §§2,5](https://arxiv.org/html/1706.06083v4) | Constrained classification inner search; MNIST/CIFAR-10 | Separate attack evaluation from excluded weight training; match checkpoint and selected paper-table attack |
| Brown et al., [patch, §§2–3](https://arxiv.org/html/1712.09665v2) | Universal replacement patch over images, locations, rotations/scales; ImageNet classifiers | Recover exact distributions, selection and model ensemble; physical evidence requires actual capture |
| Qi et al., [§§3–4; Appendix A](https://arxiv.org/html/2306.13213v2) | Corpus likelihood; released 66 targets, 40 manual requests and RealToxicityPrompts challenging evaluation | Original checkpoints, corpus sampling, preprocessing, budgets, decoding and toxicity/human protocols |
| Shayegani et al., [§§3–4; Algorithm 1](https://arxiv.org/html/2307.14539v2) | CLIP embedding matching; text/OCR/visual/combined triggers and scenario collection | Align paper and later code release; recover full trigger collection and test settings |
| Bailey et al., [§§2–4; Appendix C](https://arxiv.org/html/2309.00236v4) | Behavior and soft-distribution prompt matching; Alpaca contexts, AdvBench for jailbreak | Recover task-specific split/config; official demo is narrower than paper experiments |
| Ying et al., [§§4–5; Appendices A,D](https://arxiv.org/html/2406.04031v2) | Query-agnostic visual PGD plus iterative text feedback; paper SafetyBench scenarios and deduplicated AdvBench | Freeze original corpus/merge/splits; original ChatGPT feedback conflicts with open-only scope |
| Ling et al., [§§III–VI](https://arxiv.org/html/2601.17383v1) | Visual prompt selection/location search, Embodied City/Habitat, physical vehicle trials | Upstream incomplete-run warning; simulator, scene/camera conditions and hardware unavailable in current scope |
| Weng et al., [MMJ-Bench, setup and Tables 3–4](https://ojs.aaai.org/index.php/AAAI/article/download/34983/37138) | Unified HarmBench attack/completion scoring and MM-Vet utility | Benchmark protocol, not another individual attack; GPT-4 and open classifier are distinct evaluations |
| Wang et al., [AdaShield, §§3–4; Appendix A](https://arxiv.org/html/2403.09513v1) | Static/adaptive defense; FigStep, QR and MM-Vet | Original prompts/pool, split and retrieval/refinement; generic prefix does not reproduce this system |
| Zhang et al., [JailGuard, §§4–6](https://arxiv.org/html/2312.10766v4) | Mutations, response divergence and detection; released attack/benign collection | Match visual mutation policy, query budget, encoder, threshold and development/evaluation split |
| Zhang et al., [survey, §§3–5,7](https://arxiv.org/html/2410.23687v2) | Taxonomy and citation discovery | No separate experiment to reproduce; verify each cited original independently |

## Source findings that change implementation decisions

### Qi: use the actual parser and loop, not just README flags

Pinned [entrypoint](https://github.com/Unispac/Visual-Adversarial-Examples-Jailbreak-Large-Language-Models/blob/5fc9883cfa9b4183cf05c6eceef49e449e45e9c9/minigpt_visual_attack.py),
lines 28–32 and 114–121: the parser uses `--cfg_path`/`--gpu_id`; the attack calls
hardcode `num_iter=5000`, batch size 8, and divide alpha/epsilon by 255.
The [attacker loop](https://github.com/Unispac/Visual-Adversarial-Examples-Jailbreak-Large-Language-Models/blob/5fc9883cfa9b4183cf05c6eceef49e449e45e9c9/minigpt_utils/visual_attacker.py)
uses `range(num_iter + 1)`. Record nominal iterations and effective updates separately.
A tiny `--n_iters` argument would not make a bounded smoke run of this entrypoint.
The [released configuration](https://github.com/Unispac/Visual-Adversarial-Examples-Jailbreak-Large-Language-Models/blob/5fc9883cfa9b4183cf05c6eceef49e449e45e9c9/eval_configs/minigpt4_eval.yaml)
uses image size 224. Our 448-pixel Qwen/Smol adapter is a different deployment.

### Jailbreak in Pieces: original encoder examples are available

Pinned [source](https://github.com/erfanshayegani/Jailbreak-In-Pieces/blob/b7b2bce812adada23f583f63f63e564f84d277c6/src/adv_image.py),
lines 69–86 and 101–136: CLIP ViT-L/14-336, normalization disabled for this image
processor path, released white initialization and visual trigger, Adam learning rate
0.1, 5000 updates, pixel clamping, JPEG export. The code has a fixed loop; the paper
describes a convergence criterion. Preserve that difference rather than silently
calling one the other. Example availability does not establish complete scenario coverage.
The repository README's later LLaVA-1.5 example is also not automatically the paper model.

### Image Hijacks: demo availability is not full experiment availability

Pinned [demo config](https://github.com/euanong/image-hijacks/blob/2a4403cf67e8578afdd6e56096ff31524aaa63e7/experiments/exp_demo_imgs/config.py)
contains leak-context and specific-string jobs, full-image constraints, four epsilon
settings and LLaVA variants; it mixes Alpaca/LLaVA contexts with its own splits.
The [README](https://github.com/euanong/image-hijacks/blob/2a4403cf67e8578afdd6e56096ff31524aaa63e7/README.md)
references `exp_results_tables/config.py`, absent from the inspected pinned tree,
and says jailbreak configurations are forthcoming. Do not invent missing configurations
or substitute the demo split for the paper's split. Text formatting is evaluated only;
no actual email, tool invocation or data transmission belongs in the study.

### BAP and PPIA: constraints that cannot be fixed by renaming an adaptation

Pinned [BAP visual entrypoint](https://github.com/NY1024/BAP-Jailbreak-Vision-Language-Models-via-Bi-Modal-Adversarial-Prompt/blob/5350c0bfeb01d9e485603b85a03500280886d275/VAP.py)
sets 3000 iterations, epsilon 32/255, alpha 1/255 and batch size 2 in the attack call.
The [text demo](https://github.com/NY1024/BAP-Jailbreak-Vision-Language-Models-via-Bi-Modal-Adversarial-Prompt/blob/5350c0bfeb01d9e485603b85a03500280886d275/CoT.py)
calls GPT-3.5-turbo; a small fixed candidate list does not instantiate its feedback search.
Actual syntax checks of pinned `CoT.py` and `Eval.py` fail at lines 58 and 138,
respectively (duplicated assignment; missing function-definition colon).
Record these upstream defects; do not silently repair them and call the source
unchanged. Source identity and executable validity are separate checks.
The [PPIA README](https://github.com/2023cghacker/Physical-Prompt-Injection-Attack/blob/7291d92ee05c4b9079ef824e0b4bd12e8e14316c/README.md)
states that the code is being organized and cannot yet run fully. CyberSecEval3 image
markers are a separate benchmark, not PPIA's original simulation or physical collection.

### PGD: code release defaults need a configuration identity

Pinned [CIFAR-10 config](https://github.com/MadryLab/cifar10_challenge/blob/f15682d9f1e26eb47a2d3b371ef8b6c7abcf6276/config.json)
has epsilon 8, step size 2, 10 steps, random start and cross-entropy in that release's
pixel scale. Pinned [MNIST config](https://github.com/MadryLab/mnist_challenge/blob/3ee3643c4a8c59458d8c191b84027f4a6cbd9580/config.json)
has epsilon 0.3, step 0.01 and 40 steps. These are distinct release configurations;
do not infer that either reproduces every result table or convert scales without tracing code.

## Execution and evidence requirements

Before any new scientific run, freeze original source and environment, checkpoint and
processor hashes, source data/target hashes, original split selection, image transforms,
optimizer/update count, model sampling and judge protocol. Record a measured memory/time
profile and owner-approved budget. Keep prompts, images, raw replies and annotations private.
The original 120/60-minute clocks are expired and cannot be reset for these new studies.

No new original-paper ASR exists yet. The new notebook reports `NOT_RUN`, not zero ASR.
Only independent data/config/result review can promote a completed run to a reproduction.
Reduced scale retains an explicit sample/compute deviation; changing model, surrogate,
data, split or judge creates an adaptation even if the mathematical idea is preserved.

For comparison later: retain a separate common-benchmark lane with identical held-out
requests/images, budgets and scorer across attacks. Native-paper ASRs belong in their
own tables. Human agreement, benign utility and adaptive-defense evaluations remain required.

## Concrete next run proposal — draft, not approved or executed

Start with the released Jailbreak-in-Pieces encoder example: exact pinned source and
released initialization/trigger, original CLIP variant, original optimizer/loop and output
encoding. First audit dependency/model revisions, normalize/save-reload behavior, license,
and T4 gradients/memory. A bounded feasibility stage estimates cost; it is not a paper ASR.
Then reproduce downstream behavior with paper-compatible models and released prompts,
once their version/split differences are resolved. No manufactured triggers are permitted.

Qi original-checkpoint work follows only after weight access and memory feasibility;
do not promise the paper's A100 procedure fits a T4. Original BAP closed-model feedback
and physical capture cannot be called faithful under the current open-only/T4 scope.
Resolve those constraints explicitly; do not generate replacement corpora to hide them.
